In [1]:
library(usethis)
library(devtools)
library(roxygen2)

Warning message:
"package 'usethis' was built under R version 4.5.2"
Warning message:
"package 'devtools' was built under R version 4.5.2"
Warning message:
"package 'roxygen2' was built under R version 4.5.2"


In [2]:
has_devel()

Your system is ready to build packages!



In [3]:
library(fungioutline)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [4]:
# define functions
rank_suffix <- function() {
    # Create a table, rank name and the suffixes
    suffix <- c("aceae", "ales", "mycetidae", "mycetes", "mycota", "mycotina")
    names(suffix) <- c("Family", "Order", "Subclass", "Class", "Subphylum", "Phylum")
    suffix %>% as.data.frame() %>% rownames_to_column(var = "Rank") %>% rename(Suffix=2)-> suffix_tbl
    return(suffix_tbl)
}

In [5]:
rank_suffix()

Rank,Suffix
<chr>,<chr>
Family,aceae
Order,ales
Subclass,mycetidae
Class,mycetes
Subphylum,mycota
Phylum,mycotina


In [6]:
rank_suffix <- function() {
  # Vector of standardized suffixes for fungal taxonomic ranks
  suffix <- c("aceae", "ales", "mycetidae", "mycetes", "mycota", "mycotina")
  
  # Named vector mapping ranks to suffixes
  names(suffix) <- c("Family", "Order", "Subclass", "Class", "Phylum", "Subphylum")
  
  return(suffix)
}

In [7]:
phylum_lst <- function() {
  # Vector of standardized suffixes for fungal taxonomic ranks
  outline %>% select(Phyla) %>% filter(!grepl(pattern = "sedis", Phyla)) %>% 
    unique() %>% add_rownames("Index") %>% mutate(Index=as.integer(Index)) -> Phyla_tbl
  return(Phyla_tbl)
}

In [8]:
phylum_lst()

Warning message:
"`add_rownames()` was deprecated in dplyr 1.0.0.
ℹ Please use `tibble::rownames_to_column()` instead."


ERROR: [1m[33mError[39m in `select()`:[22m
[33m![39m Can't select columns that don't exist.
[31m✖[39m Column `Phyla` doesn't exist.


In [ ]:
summary_outline <- function(rank = "Phylum") {
  #' Count subordinate taxa within a given taxonomic rank in fungal classification
  #'
  #' @param rank Character specifying taxonomic rank to analyze ("Phylum", "Class", etc.)
  #' @return A data frame showing parent taxa and counts of their immediate subordinates
  #' @examples
  #' summary_outline("Phylum")  # Count classes per phylum
  #' summary_outline("Order")   # Count families per order
  
  # Load suffix lookup table (rank -> naming suffix)
  suffix_tbl <- rank_suffix()
  
  # Input validation - ensure requested rank is valid
  valid_ranks <- names(suffix_tbl)
  if (!rank %in% valid_ranks) {
    stop(paste("Invalid rank. Choose from:",
               paste(valid_ranks, collapse = ", ")))
  }
  
  # Get naming suffix for target rank (e.g., "-mycota" for Phylum)
  target_suffix <- suffix_tbl[rank]
  
  # DEMO DATA - Replace with your actual taxonomy data
  # Expected columns: 
  # - name: Taxon name (e.g., "Ascomycota")
  # - rank: Taxonomic rank (e.g., "Phylum")
  # - parent: Optional column for direct parent linkage
  taxa_data <- data.frame(
    name = c("Ascomycota", "Basidiomycota", "Agaricomycetes", 
             "Sordariomycetes", "Agaricales", "Pucciniales"),
    rank = c("Phylum", "Phylum", "Class", "Class", "Order", "Order")
  )
  
  # Calculate subordinate counts
  result <- taxa_data %>%
    # Filter for taxa at the rank immediately below input rank
    filter(rank == names(suffix_tbl)[which(suffix_tbl == target_suffix)]) %>%
    
    # Extract parent taxa by removing rank-specific suffix
    mutate(parent = sub(paste0(target_suffix, "$"), "", name)) %>%
    
    # Count subordinates per parent
    count(parent) %>%
    
    # Rename columns dynamically
    rename(!!rank := parent,  # Parent taxon column gets input rank name
           n_subtaxa = n)     # Standardized count column name
  
  return(result)
}

In [2]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
library(openxlsx)

In [5]:
outline <- read.xlsx('outline_2025.3.15.xlsx')

In [10]:
outline_long <- outline %>%
  mutate(Genus_syn = str_squish(Genus_syn)) %>%          # 规范空白
  separate_rows(Genus_syn, sep = "\\s*;\\s*") %>%        # 按分号拆行
  mutate(Genus_syn = na_if(Genus_syn, ""))    

In [11]:
write.xlsx(outline_long, file='outline_2025.10.20.xlsx')